# Assignment 1 - From Pixels to a Defensible Classifier
## Group 1: Training-Time Lighting Augmentation

**STUDENT IN-CLASS EVIDENCE NOTEBOOK**

Coverage: **Weeks 1–4 - Images as Data, CNNs, Evaluation/Error Analysis, Transfer Learning**.

The objective is not to maximize accuracy. The objective is to produce a **valid, reproducible and evidence-based comparison**.

In [ ]:
GROUP_NUMBER = 1
GROUP_NAME = 'Training-Time Lighting Augmentation'
SEED = 42
BASELINE_EPOCHS = 8
BASELINE_SIZE = 96
print(GROUP_NUMBER, GROUP_NAME)

## 0. Locate the offline resources
The lecturer-provided resources include the dataset, group variant, utilities, and cached source-pretrained ResNet18.

In [ ]:
from pathlib import Path
import os, sys, zipfile

def find_pack():
    candidates=[Path('/content/CV_DL_Assignment1_Practical_Pack'),Path.cwd()/ 'CV_DL_Assignment1_Practical_Pack',Path.cwd().parent/'CV_DL_Assignment1_Practical_Pack',Path.cwd(),Path.cwd().parent]
    for p in candidates:
        if (p/'utils'/'assignment_utils.py').exists() and (p/'resources').exists(): return p.resolve()
    return None
PACK=find_pack()
if PACK is None:
    try:
        from google.colab import files
        print('Upload CV_DL_Assignment1_Resources.zip supplied by the lecturer.')
        uploaded=files.upload(); zname=next((n for n in uploaded if n.lower().endswith('.zip')),None)
        if zname is None: raise RuntimeError('No ZIP uploaded')
        with zipfile.ZipFile(zname) as z: z.extractall('/content')
        PACK=find_pack()
    except Exception as e: raise RuntimeError('Practical resources not found.') from e
print('PACK =',PACK)
sys.path.insert(0,str(PACK/'utils'))
from assignment_utils import *
import pandas as pd, numpy as np, torch, matplotlib.pyplot as plt
from PIL import Image
seed_everything(SEED)
RES=PACK/'resources'; WEIGHTS=PACK/'weights'/'resnet18_source_pretrained.pth'; EVIDENCE=prepare_evidence(PACK,GROUP_NUMBER)
TRAIN_MANIFEST=RES/f'group{GROUP_NUMBER}_train_manifest.csv'; VAL_MANIFEST=RES/'common/val_manifest.csv'
print('Train manifest:',TRAIN_MANIFEST.name,'Validation:',VAL_MANIFEST.name,'Cached weights:',WEIGHTS.exists())

# Part A - Understand the Data

Before training, inspect what the model will receive. The model cannot repair an invalid data contract or a misleading split.

In [ ]:
train_df=pd.read_csv(TRAIN_MANIFEST); val_df=pd.read_csv(VAL_MANIFEST)
print('Training images:',len(train_df)); print(train_df['class_name'].value_counts().sort_index())
print(); print('Validation images:',len(val_df)); print(val_df['class_name'].value_counts().sort_index())
plot_sample_grid(RES,TRAIN_MANIFEST,EVIDENCE/'dataset_sample_grid.png',n_per_class=4,seed=SEED)
display(Image.open(EVIDENCE/'dataset_sample_grid.png'))

r=train_df.iloc[0]; im=Image.open(RES/r.path).convert('RGB'); arr=np.asarray(im)
print(); print('Example path:',r.path); print('PIL size (W,H):',im.size); print('Array shape:',arr.shape); print('dtype:',arr.dtype); print('range:',arr.min(),arr.max())

### GROUP ANSWERS - Part A

1. **Input to the system:** YOUR ANSWER  
2. **Output classes:** YOUR ANSWER  
3. **Intended use:** YOUR ANSWER  
4. **One important non-use:** YOUR ANSWER  
5. **Why classification rather than detection/segmentation?** YOUR ANSWER  
6. **One observed data/evaluation risk:** YOUR ANSWER

# Part B - Understand the CNN Before Running It

For input **3 × 96 × 96**, the baseline is:

`Conv(3→16, 3×3, pad=1) → ReLU → MaxPool(2) → Conv(16→32) → ReLU → MaxPool(2) → Conv(32→64) → ReLU → AdaptiveAvgPool(1×1) → Linear(64→3)`

**Predict first. Do not run the shape-check cell until your table is complete.**

| Stage | Your predicted output shape |
|---|---|
| Input | |
| Conv1 | |
| Pool1 | |
| Conv2 | |
| Pool2 | |
| Conv3 | |
| Adaptive Pool | |
| Linear | |

**Parameter calculation for Conv1:** YOUR WORKING

In [ ]:
model=SmallCNN(3); x=torch.zeros(4,3,96,96)
print('input',tuple(x.shape))
for layer in model.features:
    x=layer(x); print(layer.__class__.__name__,tuple(x.shape))
x=x.flatten(1); print('flatten',tuple(x.shape)); print('output',tuple(model.classifier(x).shape))
print('Total trainable parameters:',count_trainable(model))

# Part C - Establish the Baseline
All group comparisons must preserve the fixed validation set. The intervention will change only the assigned major variable.

In [ ]:
train_loader,val_loader,train_ds,val_ds=make_loaders(RES,TRAIN_MANIFEST,VAL_MANIFEST,size=BASELINE_SIZE,batch_size=64,augment=False,seed=SEED)
seed_everything(SEED); baseline_model=SmallCNN(3)
baseline_model,baseline_history,baseline_seconds=train_model(baseline_model,train_loader,val_loader,epochs=BASELINE_EPOCHS,lr=1e-3,weight_decay=0.0)
baseline_pred=predict(baseline_model,val_loader); baseline_metrics=metrics_from_predictions(baseline_pred)
print('Baseline seconds:',round(baseline_seconds,2)); print('Accuracy:',round(baseline_metrics['accuracy'],4)); print('Macro-F1:',round(baseline_metrics['macro_f1'],4)); display(baseline_metrics['per_class'].round(3))
save_learning_curves(baseline_history,EVIDENCE/'baseline_loss_curves.png','Baseline loss curves'); save_accuracy_curves(baseline_history,EVIDENCE/'baseline_accuracy_curves.png','Baseline accuracy curves')
save_confusion(baseline_pred,EVIDENCE/'baseline_confusion_matrix.png','Baseline confusion matrix'); baseline_failures=save_failure_grid(baseline_pred,RES,EVIDENCE/'baseline_failures.png',6,'Baseline: six high-confidence errors')
baseline_pred.to_csv(EVIDENCE/'baseline_predictions.csv',index=False); baseline_history.to_csv(EVIDENCE/'baseline_history.csv',index=False)

In [ ]:
display(Image.open(EVIDENCE/'baseline_loss_curves.png')); display(Image.open(EVIDENCE/'baseline_confusion_matrix.png')); display(Image.open(EVIDENCE/'baseline_failures.png'))

### GROUP ANSWERS - Baseline Evaluation

1. **Weakest class and quoted evidence:** YOUR ANSWER  
2. **Most important confusion pattern:** YOUR ANSWER  
3. **Overfit, underfit, or neither? Give TWO pieces of curve evidence:** YOUR ANSWER  
4. **Categorize the six displayed failures using the assignment taxonomy:** YOUR ANSWER  
5. **What pattern appears most often?** YOUR ANSWER  
6. **One specific next experiment suggested by the failures:** YOUR ANSWER

# Part E - Group 1 Controlled Challenge: Training-Time Lighting Augmentation
Write your hypothesis **before** running the intervention. Only the assigned major variable may change.

### GROUP HYPOTHESIS
**We predict that...** YOUR ANSWER  
**Because...** YOUR ANSWER

In [ ]:
train2,val2,_,_=make_loaders(RES,TRAIN_MANIFEST,VAL_MANIFEST,size=BASELINE_SIZE,batch_size=64,augment=True,seed=SEED)
seed_everything(SEED); challenge_model=SmallCNN(3)
challenge_model,challenge_history,challenge_seconds=train_model(challenge_model,train2,val2,epochs=BASELINE_EPOCHS,lr=1e-3)
challenge_pred=predict(challenge_model,val2)

In [ ]:
challenge_metrics=metrics_from_predictions(challenge_pred)
print('Challenge seconds:',round(challenge_seconds,2)); print('Accuracy:',round(challenge_metrics['accuracy'],4)); print('Macro-F1:',round(challenge_metrics['macro_f1'],4)); display(challenge_metrics['per_class'].round(3))
save_learning_curves(challenge_history,EVIDENCE/'challenge_loss_curves.png','Challenge loss curves'); save_confusion(challenge_pred,EVIDENCE/'challenge_confusion_matrix.png','Challenge confusion matrix'); challenge_failures=save_failure_grid(challenge_pred,RES,EVIDENCE/'challenge_failures.png',6,'Challenge: six high-confidence errors')
challenge_pred.to_csv(EVIDENCE/'challenge_predictions.csv',index=False); challenge_history.to_csv(EVIDENCE/'challenge_history.csv',index=False)
comparison=pd.DataFrame([
 {'experiment':'baseline','accuracy':baseline_metrics['accuracy'],'macro_f1':baseline_metrics['macro_f1'],'seconds':baseline_seconds},
 {'experiment':'controlled_intervention','accuracy':challenge_metrics['accuracy'],'macro_f1':challenge_metrics['macro_f1'],'seconds':challenge_seconds},
]); display(comparison.round(4)); comparison.to_csv(EVIDENCE/'baseline_vs_challenge.csv',index=False)
display(Image.open(EVIDENCE/'challenge_confusion_matrix.png')); display(Image.open(EVIDENCE/'challenge_failures.png'))

### GROUP ANSWERS - Controlled Experiment

**Required diagnostic:** Compare validation class recalls and the six failure images. Did lighting/exposure-related errors become less common?

1. **Was the hypothesis supported, partly supported, or not supported? Quote numerical evidence.** YOUR ANSWER  
2. **What improved?** YOUR ANSWER  
3. **What became worse or remained weak?** YOUR ANSWER  
4. **Why is this a controlled comparison?** YOUR ANSWER  
5. **What uncertainty remains after one run?** YOUR ANSWER

# Part F - Transfer Learning
The cached ResNet18 was **pretrained on a separate six-class source task** (`bottle`, `cup`, `sheet`, `carton`, `can`, `cap`). The assignment target task merges those subtypes into `plastic`, `paper`, and `metal`.

This lets you directly explain what transfer learning means: the backbone begins with parameters learned on a related source task rather than random initialization.

In [ ]:
transfer_train,transfer_val,_,_=make_loaders(RES,TRAIN_MANIFEST,VAL_MANIFEST,size=96,batch_size=64,augment=False,seed=SEED,transfer=True)
transfer_model,source_ckpt=load_source_pretrained_resnet(WEIGHTS,num_target_classes=3,freeze_backbone=True)
print('Source classes:',source_ckpt['source_classes']); print('All parameters:',count_all(transfer_model)); print('Trainable parameters:',count_trainable(transfer_model)); print('New head:',transfer_model.fc)
seed_everything(SEED)
transfer_model,transfer_history,transfer_seconds=train_model(transfer_model,transfer_train,transfer_val,epochs=3,lr=2e-3)
transfer_pred=predict(transfer_model,transfer_val); transfer_metrics=metrics_from_predictions(transfer_pred)
print('Transfer seconds:',round(transfer_seconds,2)); print('Accuracy:',round(transfer_metrics['accuracy'],4)); print('Macro-F1:',round(transfer_metrics['macro_f1'],4)); display(transfer_metrics['per_class'].round(3))
save_learning_curves(transfer_history,EVIDENCE/'transfer_loss_curves.png','Transfer learning loss curves'); save_confusion(transfer_pred,EVIDENCE/'transfer_confusion_matrix.png','Transfer learning confusion matrix'); save_failure_grid(transfer_pred,RES,EVIDENCE/'transfer_failures.png',6,'Transfer model: six high-confidence errors')
transfer_pred.to_csv(EVIDENCE/'transfer_predictions.csv',index=False); transfer_history.to_csv(EVIDENCE/'transfer_history.csv',index=False)

### GROUP ANSWERS - Transfer Learning

1. **What was transferred?** YOUR ANSWER  
2. **What is the backbone?** YOUR ANSWER  
3. **What is the task-specific head?** YOUR ANSWER  
4. **Does freezing remove the backbone from the forward pass? Explain.** YOUR ANSWER  
5. **Why is normalization used in the transfer pipeline?** YOUR ANSWER  
6. **Did transfer learning improve this target task under the fixed validation protocol? Quote evidence and mention one remaining failure.** YOUR ANSWER

# Part G - Final Evidence Lock
Run the next cell **before leaving the laboratory**. It saves a comparison CSV and creates the ZIP that must be submitted with the notebook.

In [ ]:
summary_rows=[
 {'experiment':'baseline_cnn','trainable_params':count_trainable(baseline_model),'key_change':'baseline','accuracy':baseline_metrics['accuracy'],'macro_f1':baseline_metrics['macro_f1'],'seconds':baseline_seconds},
 {'experiment':'controlled_intervention','trainable_params':count_trainable(challenge_model),'key_change':GROUP_NAME,'accuracy':challenge_metrics['accuracy'],'macro_f1':challenge_metrics['macro_f1'],'seconds':challenge_seconds},
 {'experiment':'transfer_resnet18_head','trainable_params':count_trainable(transfer_model),'key_change':'cached source-pretrained backbone; new frozen-backbone head','accuracy':transfer_metrics['accuracy'],'macro_f1':transfer_metrics['macro_f1'],'seconds':transfer_seconds},
]
save_experiment_summary(EVIDENCE,summary_rows)
summary=pd.DataFrame(summary_rows); display(summary.round(4))
zip_path=zip_evidence(EVIDENCE,PACK/f'Group_{GROUP_NUMBER}_Evidence.zip')
print('Evidence ZIP:',zip_path)
print('Submit this notebook and the evidence ZIP before leaving class.')

## Final group conclusion

**Which approach would your group continue developing and why?** YOUR ANSWER  

**Three limitations of this in-class experiment:**  
1. YOUR ANSWER  
2. YOUR ANSWER  
3. YOUR ANSWER  

**One specific next experiment justified by the observed failures:** YOUR ANSWER